# YOLO11n-ADAS-v2: Empirical Domain Adaptation and Multi-Metric Calibration
## Data-Centric Optimization for Resource-Constrained Automotive Edge Systems

### 1. Abstract and Theoretical Formulation
This computational notebook establishes the formal implementation of **YOLO11n-ADAS-v2**, an empirical domain-adapted object detection pipeline designed for automotive advanced driver assistance systems (ADAS). Operating detection transformers or convolutional neural networks within vehicle cabins entails a fundamental trade-off governed by:
1. **Temporal Latency Constraints:** Vehicle displacement at operating speed ($v = 80\text{ km/h} \approx 22.2\text{ m/s}$) imposes a strict deadline on per-frame inference latency ($\le 200\text{ ms}$). Protracted inference cycles diminish spatial observation density, preventing timely temporal confirmation.
2. **Acoustic Driver Fatigue Mitigation:** In safety-critical embedded alert systems, Type I errors (False Positives) induce cognitive alarm fatigue. Minimizing empirical risk requires robust background discrimination across variable illumination and asphalt geometries.

### 2. Methodological Principles
- **Pretrained Feature Transfer:** Initializing weights from MS-COCO (`yolo11n.pt`) to retain generalized spatial feature extractors prior to task-specific parameter fine-tuning.
- **Empirical Negative Evidence Ingestion:** Incorporating 186 field-recorded non-target patches from on-road driving trials in Jazan (cantilever brackets, light poles, asphalt texture anomalies, reverse sign faces) registered as native zero-annotation background images.
- **Domain-Constrained Augmentation:** Enforcing $\text{fliplr} = 0.0$ and $\text{mixup} = 0.0$ to preserve semantic fidelity for directional cues and numeral speed values, while introducing a 20-epoch mosaic deactivation phase ($\text{close\_mosaic} = 20$) for realistic final-stage convergence.
- **Deployment Candidate Selection:** Execution across 150 epochs with periodic 10-epoch checkpoint retention, followed by holistic multi-metric Pareto profiling (Precision, Recall, F1, FP, FN, mAP50, and mAP50-95).

In [ ]:
# ==============================================================================
# SECTION 1: PERSISTENT STORAGE INITIALIZATION & DEPENDENCY SETUP
# ==============================================================================
import os
import sys
import shutil
import zipfile
import yaml
import logging
from pathlib import Path
from google.colab import drive

# Configure standard academic logging
logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger("YOLO11_ADAS_v2")

logger.info("Mounting Google Drive storage interface...")
drive.mount("/content/drive")

# Install required dependencies for training, graph simplification, and quantization
!pip install -q ultralytics onnx onnxsim onnxruntime onnxconverter-common

# Initialize persistent target directory on Google Drive
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/YOLO11_ADAS_v2")
DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
logger.info("Persistent project directory validated: %s", DRIVE_PROJECT_DIR)


Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 121.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 11.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have pro

In [ ]:
# ==============================================================================
# SECTION 2: HIGH-THROUGHPUT SCRATCHPAD CACHING & HARD NEGATIVE INTEGRATION
# ==============================================================================
LOCAL_DATASET = Path("/content/dataset")

# Locate primary training dataset archive across Google Drive storage paths
DATASET_ARCHIVE_CANDIDATES = [
    Path("/content/drive/MyDrive/Offline Arabic ADAS.yolov11.zip"),
    Path("/content/drive/MyDrive/DFINE_Training/Offline Arabic ADAS.yolov11.zip"),
    Path("/content/drive/MyDrive/Offline_Arabic_ADAS_yolov11.zip")
]

dataset_zip_path = next((p for p in DATASET_ARCHIVE_CANDIDATES if p.is_file()), None)
if dataset_zip_path is None:
    search_results = list(Path("/content/drive/MyDrive").glob("*yolo*.zip")) + list(Path("/content/drive/MyDrive").glob("*ADAS*.zip"))
    if search_results:
        dataset_zip_path = search_results[0]

if dataset_zip_path is None:
    raise FileNotFoundError("Primary dataset archive (.zip) could not be located in Google Drive.")

logger.info("Extracting primary dataset to local high-speed storage: %s", dataset_zip_path)
if not (LOCAL_DATASET / "train").exists():
    with zipfile.ZipFile(dataset_zip_path, 'r') as zf:
        zf.extractall(LOCAL_DATASET)
    logger.info("Dataset extraction complete.")

train_images_dir = LOCAL_DATASET / "train" / "images"
train_labels_dir = LOCAL_DATASET / "train" / "labels"
train_labels_dir.mkdir(parents=True, exist_ok=True)

# Ingest 186 empirical hard negative patches collected during field evaluations
# In YOLO format, non-target images with 0-byte annotation files act as explicit negative evidence
NEGATIVE_ARCHIVE_CANDIDATES = [
    Path("/content/drive/MyDrive/DFINE_Training/extra_negatives.zip"),
    Path("/content/drive/MyDrive/extra_negatives.zip")
]
negatives_zip_path = next((p for p in NEGATIVE_ARCHIVE_CANDIDATES if p.is_file()), None)
if negatives_zip_path is None:
    neg_matches = list(Path("/content/drive/MyDrive").glob("*negative*.zip"))
    if neg_matches:
        negatives_zip_path = neg_matches[0]

if negatives_zip_path and negatives_zip_path.is_file():
    temp_extract_dir = Path("/content/temp_negatives")
    temp_extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(negatives_zip_path, 'r') as zf:
        zf.extractall(temp_extract_dir)

    negative_image_paths = sorted(list(temp_extract_dir.glob("*.jpg")) + list(temp_extract_dir.glob("*.png")))
    logger.info("Located %d empirical negative samples. Registering as background images...", len(negative_image_paths))

    for index, sample_path in enumerate(negative_image_paths):
        target_name = f"hard_neg_field_{index:04d}.jpg"
        shutil.copy2(sample_path, train_images_dir / target_name)

        # Create empty annotation file to penalize false positive activations
        empty_txt = train_labels_dir / target_name.replace(".jpg", ".txt").replace(".png", ".txt")
        empty_txt.touch(exist_ok=True)

    logger.info("Successfully registered %d background images with zero-byte label files.", len(negative_image_paths))
else:
    logger.warning("Empirical negative archive not identified; proceeding with baseline dataset background distribution.")

# Verify and configure dataset specification YAML
yaml_candidates = list(LOCAL_DATASET.rglob("data.yaml"))
if not yaml_candidates:
    raise FileNotFoundError("Configuration file 'data.yaml' not located in extracted dataset.")
data_yaml_path = yaml_candidates[0]

with open(data_yaml_path, 'r', encoding='utf-8') as f:
    data_spec = yaml.safe_load(f)

data_spec['path'] = str(LOCAL_DATASET.resolve())
data_spec['train'] = "train/images"
data_spec['val'] = "valid/images"

with open(data_yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data_spec, f, sort_keys=False)

logger.info("Dataset configuration successfully finalized: %s", data_yaml_path)
logger.info("Categories Detected (%d): %s", data_spec.get('nc', 14), data_spec.get('names'))


In [ ]:
# ==============================================================================
# SECTION 3: DOMAIN-CONSTRAINED TRAINING PIPELINE EXECUTION (150 EPOCHS)
# ==============================================================================
from ultralytics import YOLO

# Instantiate official MS-COCO pretrained weights for transfer learning
model = YOLO("yolo11n.pt")

logger.info("=" * 80)
logger.info("COMMENCING DOMAIN-CONSTRAINED FINE-TUNING: YOLO11n-ADAS-v2")
logger.info("Epochs: 150 | Patience: 150 (Guarantees uninterrupted execution across 150 epochs)")
logger.info("Persistence: save_period=10 (Continuous checkpoint serialization directly to Google Drive)")
logger.info("Augmentation Constraints: fliplr=0.0 (Preserves asymmetry and numerals), mixup=0.0")
logger.info("Feature Consolidation: close_mosaic=20 (Unmodified camera imagery during final 20 epochs)")
logger.info("=" * 80)

# Execute model training with direct Google Drive persistence
training_results = model.train(
    # Data and Spatial Geometry
    data=str(data_yaml_path),
    imgsz=640,
    batch=16,
    epochs=150,

    # Execution Schedule & Checkpoint Retention Policy
    patience=150,                     # Enforces execution through all 150 epochs without early termination
    save_period=10,                   # Serializes epoch10.pt, epoch20.pt, ..., epoch150.pt
    project=str(DRIVE_PROJECT_DIR),   # Stream weights and telemetry directly to Google Drive
    name="run_yolo11n_adas_v2",
    exist_ok=True,

    # Optimization and Learning Rate Schedule
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,                      # Cosine Annealing learning rate decay
    warmup_epochs=3.0,

    # Domain-Constrained Augmentation Specification
    fliplr=0.0,                       # Disabled: Prevents semantic corruption of numeral & directional signs
    flipud=0.0,                       # Disabled: Vertical inversion is non-physical in road geometry
    mixup=0.0,                        # Disabled: Eliminates synthetic ghost sign superposition
    mosaic=1.0,
    close_mosaic=20,                  # Deactivates mosaic augmentation during the final 20 epochs

    # Photometric, Tilt and Scale Augmentation
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,                        # Simulates variable solar elevation, glare, and shadow profiles
    degrees=3.0,                      # Simulates slight vehicle chassis pitch and camera mount tilt
    translate=0.1,
    scale=0.35,                       # Simulates sign approach distance variation [0.65x - 1.35x]

    # Hardware Acceleration
    device=0,
    workers=4,
    amp=True
)

logger.info("Training execution completed across 150 epochs.")
logger.info("Model artifacts available in: %s", DRIVE_PROJECT_DIR / "run_yolo11n_adas_v2" / "weights")


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.171 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=20, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/dataset/data.yaml, degrees=3.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, fre

In [ ]:
# ==============================================================================
# SECTION 4: FAULT-TOLERANT RESUMPTION UTILITY (EXECUTE ONLY UPON DISCONNECTION)
# ==============================================================================
# If Google Colab terminates due to network failure or GPU instance recycling,
# execute this cell to resume training from the exact state saved in last.pt.
from ultralytics import YOLO
from pathlib import Path

LAST_CHECKPOINT_PATH = Path("/content/drive/MyDrive/YOLO11_ADAS_v2/run_yolo11n_adas_v2/weights/last.pt")
if LAST_CHECKPOINT_PATH.is_file():
    print(f"Resuming model training from persistent state: {LAST_CHECKPOINT_PATH}")
    checkpoint_model = YOLO(str(LAST_CHECKPOINT_PATH))
    checkpoint_model.train(resume=True)
else:
    print("Prior checkpoint 'last.pt' not found. Launch initial training via Section 3.")


In [ ]:
# ==============================================================================
# SECTION 5: DEPLOYMENT CANDIDATE SELECTION (PARETO PROFILING)
# ==============================================================================
# NOTE: Offline confusion-matrix False Positives (FP) serve as an indicative surrogate.
# Conclusive deployment suitability is validated via live in-vehicle driving trials (FP/min).
import os
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

WEIGHTS_DIRECTORY = Path("/content/drive/MyDrive/YOLO11_ADAS_v2/run_yolo11n_adas_v2/weights")
checkpoints_to_profile = sorted(
    list(WEIGHTS_DIRECTORY.glob("epoch*.pt")),
    key=lambda item: int(''.join(filter(str.isdigit, item.stem))) if any(char.isdigit() for char in item.stem) else 0
)
if (WEIGHTS_DIRECTORY / "best.pt").is_file():
    checkpoints_to_profile.append(WEIGHTS_DIRECTORY / "best.pt")
if (WEIGHTS_DIRECTORY / "last.pt").is_file():
    checkpoints_to_profile.append(WEIGHTS_DIRECTORY / "last.pt")

profile_records = []
print(f"Initiating standardized multi-metric evaluation across {len(checkpoints_to_profile)} checkpoints...")

for ckpt_path in checkpoints_to_profile:
    try:
        evaluation_model = YOLO(str(ckpt_path))
        metrics = evaluation_model.val(data=str(data_yaml_path), split="val", imgsz=640, batch=16, verbose=False)

        # Extract confusion matrix for structural classification breakdown
        # Ultralytics convention: axis 0 (rows) = Predicted, axis 1 (cols) = True GT
        # Final index (-1) = background
        confusion_mat = metrics.confusion_matrix.matrix
        fn_total = int(confusion_mat[-1, :-1].sum())       # True hazard predicted as background (Missed Hazard / FN)
        fp_total = int(confusion_mat[:-1, -1].sum())       # True background predicted as hazard (False Positive / BgFP)
        tp_total = int(confusion_mat[:-1, :-1].diagonal().sum())

        mean_precision = metrics.box.mp
        mean_recall = metrics.box.mr
        f1_metric = (2.0 * mean_precision * mean_recall) / (mean_precision + mean_recall + 1e-8)
        map_50 = metrics.box.map50
        map_50_95 = metrics.box.map

        profile_records.append({
            "Checkpoint": ckpt_path.name,
            "Precision": round(mean_precision * 100.0, 2),
            "Recall": round(mean_recall * 100.0, 2),
            "F1-Score": round(f1_metric * 100.0, 2),
            "BgFP (Raw Val Low-Conf)": fp_total,
            "FN (Missed Hazards)": fn_total,
            "TP": tp_total,
            "mAP50": round(map_50 * 100.0, 2),
            "mAP50-95": round(map_50_95 * 100.0, 2)
        })
    except Exception as exc:
        print(f"Warning: Evaluation failed for {ckpt_path.name}: {exc}")

df_selection = pd.DataFrame(profile_records)
print("\n" + "=" * 95)
print("DEPLOYMENT CANDIDATE SELECTION TABLE (OFFLINE PERFORMANCE AUDIT)")
print("=" * 95)
print(df_selection.to_string(index=False))

# Persist candidate selection table to Google Drive
report_output_path = DRIVE_PROJECT_DIR / "deployment_candidates_report.csv"
df_selection.to_csv(report_output_path, index=False)
print(f"\nCandidate selection report saved: {report_output_path}")

Initiating standardized multi-metric evaluation across 17 checkpoints...
Ultralytics 8.4.171 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 100 layers, 2,584,882 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 435.8±249.7 MB/s, size: 10.8 KB)
val: Scanning /content/dataset/valid/labels.cache... 1767 images, 321 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1767/1767 570.1Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 2, len(boxes) = 2405. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 111/111 5.5it/s 20.1s
                   all       1767       2405      0.592      0.403      0.421      0.263
Speed: 1.1ms preprocess, 4.3ms inference, 0.0ms l

In [ ]:
# ==============================================================================
# SECTION 6: MULTI-PRECISION ONNX COMPILATION FOR MOBILE ADAS ENGINE
# ==============================================================================
# Compiles the designated Pareto candidate checkpoint to FP32, Dynamic INT8, and FP16
import onnx
import shutil
from pathlib import Path
from ultralytics import YOLO
from onnxconverter_common import float16
from onnxruntime.quantization import quantize_dynamic, QuantType

# Specify nominated candidate checkpoint (Default: best.pt or verified Pareto candidate)
CHOSEN_CANDIDATE = Path("/content/drive/MyDrive/YOLO11_ADAS_v2/run_yolo11n_adas_v2/weights/best.pt")
assert CHOSEN_CANDIDATE.is_file(), f"Candidate checkpoint not found at: {CHOSEN_CANDIDATE}"

EXPORT_TARGET_DIR = DRIVE_PROJECT_DIR / "exported_models"
EXPORT_TARGET_DIR.mkdir(parents=True, exist_ok=True)

print(f"Compiling candidate weights: {CHOSEN_CANDIDATE}")
yolo_export = YOLO(str(CHOSEN_CANDIDATE))

# Step A: Export Base FP32 ONNX Graph with Graph Simplification
onnx_export_path = yolo_export.export(
    format="onnx",
    imgsz=640,
    dynamic=False,
    simplify=True,
    opset=17
)

fp32_destination = EXPORT_TARGET_DIR / "yolo11n_adas_v2_640.onnx"
shutil.copy2(onnx_export_path, fp32_destination)
print(f"[1/3] FP32 Baseline Compiled:  {fp32_destination} ({fp32_destination.stat().st_size / (1024*1024):.2f} MB)")

# Step B: Dynamic Asymmetric INT8 Quantization (Optimized for Mobile ARM NEON / SIMD)
int8_destination = EXPORT_TARGET_DIR / "yolo11n_adas_v2_640_int8.onnx"
try:
    quantize_dynamic(
        model_input=str(fp32_destination),
        model_output=str(int8_destination),
        weight_type=QuantType.QInt8
    )
    print(f"[2/3] INT8 Quantized Model:    {int8_destination} ({int8_destination.stat().st_size / (1024*1024):.2f} MB)")
except Exception as err:
    print(f"Warning: INT8 dynamic quantization failed: {err}")

# Step C: IEEE 754 Half-Precision (FP16) Transformation (Mobile GPU / NNAPI Target)
fp16_destination = EXPORT_TARGET_DIR / "yolo11n_adas_v2_640_fp16.onnx"
try:
    raw_proto = onnx.load(str(fp32_destination))
    fp16_proto = float16.convert_float_to_float16(raw_proto, keep_io_types=True)
    onnx.save(fp16_proto, str(fp16_destination))
    print(f"[3/3] FP16 Model Compiled:     {fp16_destination} ({fp16_destination.stat().st_size / (1024*1024):.2f} MB)")
except Exception as err:
    print(f"Warning: FP16 conversion failed: {err}")

print("=" * 75)
print(f"Deployment package finalized in: {EXPORT_TARGET_DIR}")
print("=" * 75)


Compiling candidate weights: /content/drive/MyDrive/YOLO11_ADAS_v2/run_yolo11n_adas_v2/weights/best.pt
Ultralytics 8.4.171 🚀 Python-3.13.15 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11n summary (fused): 100 layers, 2,584,882 parameters, 0 gradients, 6.4 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/YOLO11_ADAS_v2/run_yolo11n_adas_v2/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 18, 8400) (5.2 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 10 packages in 381ms
Prepared 2 packages in 36ms
Installed 2 packages in 6ms
 + colorama==0.4.6
 + onnxslim==0.1.97

requirements: AutoUpdate success ✅ 0.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting

[1/3] FP32 Baseline Compiled:  /content/drive/MyDrive/YOLO11_ADAS_v2/exported_models/yolo11n_adas_v2_640.onnx (10.12 MB)
[2/3] INT8 Quantized Model:    /content/drive/MyDrive/YOLO11_ADAS_v2/exported_models/yolo11n_adas_v2_640_int8.onnx (2.88 MB)
[3/3] FP16 Model Compiled:     /content/drive/MyDrive/YOLO11_ADAS_v2/exported_models/yolo11n_adas_v2_640_fp16.onnx (5.11 MB)
Deployment package finalized in: /content/drive/MyDrive/YOLO11_ADAS_v2/exported_models


/usr/local/lib/python3.13/dist-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -8.131438988812079e-08 will be truncated to -1e-07
  warnings.warn(


### 7. Held-Out Unseen Test Split Evaluation & Operating-Point Audit (`conf=0.38`)
Evaluates `best.pt` on the completely unseen `split="test"` (884 images, 1,278 annotated instances) under both standard PR sweep (`conf=0.001`) and the deployed Android threshold (`conf=0.38`).

In [ ]:
# ==============================================================================
# SECTION 7: UNSEEN TEST SPLIT EVALUATION & OPERATING POINT AUDIT
# ==============================================================================
from pathlib import Path
from ultralytics import YOLO

BEST_WEIGHTS = Path('/content/drive/MyDrive/YOLO11_ADAS_v2/run_yolo11n_adas_v2/weights/best.pt')
eval_model = YOLO(str(BEST_WEIGHTS))

# 7A. Standard PR-sweep on unseen test split (conf=0.001 default)
test_default = eval_model.val(data=str(data_yaml_path), split='test', imgsz=640, batch=16, plots=True)
print(f'Unseen Test Default -> P={test_default.box.mp*100:.2f}%, R={test_default.box.mr*100:.2f}%, mAP50={test_default.box.map50*100:.2f}%, mAP50-95={test_default.box.map*100:.2f}%')

# 7B. Deployed Android Operating Point on unseen test split (conf=0.38)
test_op = eval_model.val(data=str(data_yaml_path), split='test', conf=0.38, imgsz=640, batch=16, plots=True)
cm = test_op.confusion_matrix.matrix
tp_op = int(cm[:-1, :-1].diagonal().sum())
fn_op = int(cm[-1, :-1].sum())
fp_op = int(cm[:-1, -1].sum())
inter_err = int(cm[:-1, :-1].sum()) - tp_op
print(f'Unseen Test @conf=0.38 -> TP={tp_op}, FN(Missed)={fn_op}, BgFP={fp_op}, InterClassErr={inter_err}')